<a href="https://colab.research.google.com/github/AkkiNikumbh/DL-EXPERIMENTS/blob/main/Bi_Directional_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout

# Hyperparameters
vocab_size = 10000
max_length = 200
embedding_dim = 128

# Load dataset
print("Loading data...")
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=vocab_size)

# Pad sequences to ensure uniform input size
x_train = pad_sequences(x_train, maxlen=max_length)
x_test = pad_sequences(x_test, maxlen=max_length)

print(f"Training data shape: {x_train.shape}")

Loading data...
17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Training data shape: (25000, 200)


In [ ]:
model = Sequential([
    # Embedding layer to convert word indices to dense vectors
    Embedding(vocab_size, embedding_dim, input_length=max_length),

    # Bi-Directional LSTM layer
    Bidirectional(LSTM(64, return_sequences=False)),

    # Regularization and Output
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid') # Binary output: 0 (negative) or 1 (positive)
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
batch_size = 64
epochs = 5

history = model.fit(
    x_train, y_train,
    batch_size=batch_size,
    epochs=epochs,
    validation_data=(x_test, y_test)
)

Epoch 1/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 236s 590ms/step - accuracy: 0.7978 - loss: 0.4263 - val_accuracy: 0.8491 - val_loss: 0.3513
Epoch 2/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 227s 582ms/step - accuracy: 0.9043 - loss: 0.2468 - val_accuracy: 0.8712 - val_loss: 0.3074
Epoch 3/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 257s 658ms/step - accuracy: 0.9298 - loss: 0.1903 - val_accuracy: 0.8652 - val_loss: 0.3401
Epoch 4/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 232s 582ms/step - accuracy: 0.9399 - loss: 0.1593 - val_accuracy: 0.8621 - val_loss: 0.3784
Epoch 5/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 291s 657ms/step - accuracy: 0.9620 - loss: 0.1050 - val_accuracy: 0.8470 - val_loss: 0.4431


In [ ]:
loss, accuracy = model.evaluate(x_test, y_test)
print(f"\nTest Accuracy: {accuracy*100:.2f}%")

# Quick helper to test a custom review
def predict_sentiment(text):
    word_index = imdb.get_word_index()
    # Tokenize and encode the text based on the IMDB index
    tokens = [word_index.get(w.lower(), 0) for w in text.split()]
    tokens = [t if t < vocab_size else 0 for t in tokens]
    padded = pad_sequences([tokens], maxlen=max_length)

    prediction = model.predict(padded)[0][0]
    return "Positive" if prediction > 0.5 else "Negative"

# Example usage:
# print(predict_sentiment("this movie was absolutely fantastic and brilliant"))

782/782 ━━━━━━━━━━━━━━━━━━━━ 49s 62ms/step - accuracy: 0.8470 - loss: 0.4431

Test Accuracy: 84.70%
